# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, following the Croissant schema.

### Dataset Source
The dataset schema is described at:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their IDs.

**Note:** All entities are referenced by their `@id`.

In [ ]:
# List all record sets by @id and name

record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the Croissant schema.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- {rs['@id']} - {rs.get('name','(no name)')}")
    print()

    # Show fields in each record set by @id and name
    for rs in record_sets:
        print(f"Fields for Record Set: {rs['@id']}")
        for field in rs.get('field', []):
            print(f"    Field: {field['@id']} - {field.get('name','(no name)')} ({field.get('dataType','unknown type')})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

For this dataset, we'll extract the primary patient-level table which is present in the schema's single main record set. All fields and columns will be referenced by their `@id`.

In [ ]:
# Identify the main record set (use @id for referencing)
main_record_set = None
if record_sets:
    main_record_set = record_sets[0]['@id']
    print(f"Selected main record set: {main_record_set}")
else:
    raise ValueError("No record sets found.")

# Load all records from the main record set
records = list(dataset.records(record_set=main_record_set))
df = pd.DataFrame(records)

print(f"Columns (fields by @id) in DataFrame:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's explore and process the dataset with common data operations:
- Filter: Select individuals above a certain age (age is personally sensitive, see [personalSensitiveInformation])
- Normalize: Standardize the age column
- Group by: Anatomical location of second primary CRC (`anatomical_location_2`, if present)

> **Note:** All operations reference columns and fields by their schema `@id`.

In [ ]:
# Identify a numeric field and a grouping field by @id from the schema info above
# We'll attempt to choose 'age' for numeric filtering and normalization;
# for group, use 'anatomical_location_2' if present. Adjust if needed.

# Find likely '@id's by examining the DataFrame columns
print("Available columns:", df.columns.tolist())

# Let's try to use '@id' columns: choose the one for age and one for anatomical location
# Assume 'age' appears literally (adjust if schema has e.g. cr:age or similar)

numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: just pick first numeric-like field
    for col in df.columns:
        if df[col].dtype in [int, float] or df[col].astype(str).str.isnumeric().any():
            numeric_field_id = col
            break
print(f"Numeric field selected: {numeric_field_id}")

group_field_id = None
for col in df.columns:
    if 'anatomical' in col.lower():
        group_field_id = col
        break
print(f"Grouping field selected: {group_field_id}")

# Apply basic filters/EDA
if numeric_field_id in df.columns:
    # Convert age to numeric in case it's string
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 60  # e.g., elderly patients
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Numeric field {numeric_field_id} not found in columns.")

# Group by anatomical location if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships using fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of age
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id} in dataset')
    plt.show()

# Box plot by anatomical location
if group_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f'{numeric_field_id} grouped by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset on second primary colorectal cancer survivors using the `mlcroissant` library. All tables, fields, and analysis steps referenced the schema's unique `@id` values. Key clinical attributes such as patient age and cancer anatomical location were normalized and visualized, demonstrating schema-aware EDA. For additional work, this methodology enables fully reproducible, FAIR data workflows linking code directly to metadata-driven identifiers.